<a href="https://colab.research.google.com/github/tharun0223darla/flyrank-search-intelligence/blob/main/Copy_of_w01_research_question.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tharun0223darla/flyrank-search-intelligence/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

## 1. My lane and why

My provisional choice is **Lane 2 — Refresh / Content Opportunity
Scoring**. I chose this lane because content teams have limited review
capacity and need a defensible way to decide which pages deserve attention
first. The starter notebooks showed that observable signals such as
impressions, position, age, freshness, CTR, and engagement can support a
ranked review queue. My unit of analysis will be one pseudonymized content
page. The proposed output will be a ranked list of pages with a priority
score, reason codes, and a suggested review action.

This lane is provisional. I may refine its target after examining the
warehouse's daily history and defining non-overlapping feature and future
outcome windows.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [ ]:
# Load the starter repository and dataset safely in Colab.

from pathlib import Path
import os
import sys
import subprocess
import pandas as pd

REPO_URL = "https://github.com/tharun0223darla/flyrank-search-intelligence.git"
REPO_DIR = "flyrank-search-intelligence"

if "google.colab" in sys.modules:
    repo_path = Path("/content") / REPO_DIR

    if not repo_path.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", REPO_URL, str(repo_path)],
            check=True
        )

    os.chdir(repo_path)

data_path = Path("data/raw/content_refresh_anonymized.csv")

assert data_path.exists(), (
    "Starter dataset not found. Make sure the notebook is running "
    "from the repository root."
)

df = pd.read_csv(data_path)

print(f"Loaded {len(df):,} pseudonymized content pages.")
print(f"Columns available: {df.shape[1]}")

Loaded 30,000 pseudonymized content pages.
Columns available: 44


## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

## 2. The question: decision, action, cost of a wrong call

**Research question:** Among pages with enough observed demand to justify
attention, which pages should a content or SEO reviewer inspect first for
refresh, protection, expansion, consolidation, pruning, or monitoring?

**Decision:** The work improves the decision about how to allocate a limited
page-review capacity.

**Actor and action:** A content editor or SEO analyst receives a ranked queue,
examines the reason codes and supporting signals, and then decides whether to
refresh, protect, expand, consolidate, prune, or monitor each page. The system
provides decision support; it does not make the final editorial decision.

**Unit of analysis:** One pseudonymized content page.

**Output:** A ranked page-review queue containing a priority score, confidence
label, reason codes, and a suggested human-review action.

**Cost of a wrong recommendation:** A false positive can waste editorial time
or encourage an unnecessary change to a healthy page. A false negative can
leave a valuable declining or underperforming page unreviewed. Because the
team can only inspect a limited number of pages, ranking quality near the top
of the queue matters most.

**Why data or ML may help:** Page opportunity can depend on interactions among
visibility, position, age, freshness, CTR, engagement, and content type. These
patterns may be too complex for one fixed rule. I will still begin with a
transparent rule baseline and only use a more complex model if client-holdout
or time-aware validation shows that it improves the review queue while
retaining understandable reason codes.

A suitable provisional evaluation metric is **Precision@50**, assuming a
reviewer can inspect approximately 50 pages in one review cycle.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [ ]:
# Provisional decision policy; these are explainable starting assumptions.

review_capacity = 50
minimum_demand = 100

print(f"Provisional review capacity: {review_capacity} pages")
print(f"Initial minimum-demand threshold: {minimum_demand} impressions")
print("Provisional ranking metric: Precision@50")

Provisional review capacity: 50 pages
Initial minimum-demand threshold: 100 impressions
Provisional ranking metric: Precision@50


## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

## 3. Quick look at the data (2–3 real numbers)

I computed three aggregate indicators from the starter dataset. They do not
prove that a page should be changed, but they show that the inventory contains
enough observable decline, visibility, and CTR-review context to justify
studying a page-prioritization system.

In [ ]:
# Aggregate, public-safe indicators supporting the provisional lane.

declining = df["trend_direction"].str.lower().eq("down")

declining_with_demand = (
    declining
    & df["impressions_90d"].ge(100)
)

visible_low_ctr = (
    df["impressions_90d"].ge(500)
    & df["avg_position"].gt(0)
    & df["avg_position"].le(20)
    & df["ctr"].lt(0.5)  # CTR is already on a 0–100 percentage scale
)

older_page_one_pages = (
    df["avg_position"].gt(0)
    & df["avg_position"].le(10)
    & df["content_age_days"].ge(180)
)

def report(name, mask):
    count = int(mask.sum())
    percentage = 100 * mask.mean()
    print(f"{name}: {count:,} pages ({percentage:.2f}% of the dataset)")

report("Declining pages with at least 100 impressions", declining_with_demand)
report("Visible position 1–20 pages with CTR below 0.5%", visible_low_ctr)
report("Pages at positions 1–10 that are at least 180 days old", older_page_one_pages)

Declining pages with at least 100 impressions: 13,152 pages (43.84% of the dataset)
Visible position 1–20 pages with CTR below 0.5%: 9,759 pages (32.53% of the dataset)
Pages at positions 1–10 that are at least 180 days old: 7,076 pages (23.59% of the dataset)


These aggregates suggest that the dataset contains a substantial number of
pages with observable demand and review-relevant signals. In particular,
13,152 pages were both classified as declining in the current starter window
and had at least 100 impressions. There were also 9,759 visible pages in
positions 1–20 with CTR below 0.5%, and 7,076 older pages appearing in
positions 1–10.

These groups overlap and should not be added together. They are candidate
contexts for further analysis, not confirmed recommendations. The next stages
should determine whether a transparent ranking rule—and eventually a
validated model—can prioritize useful candidates more effectively.

## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

## 4. Careful words: what I can and can't claim

This project may identify **observed associations**, compare ranking methods,
and produce **directional, decision-support recommendations** about which
pseudonymized pages deserve human review first. It may also measure whether a
ranking method improves top-K selection against a transparent baseline under
client-holdout or time-aware validation.

It cannot prove that refreshing a page caused or will cause recovery. It
cannot claim to predict Google's algorithm, identify a Google ranking factor,
or guarantee that a suggested action will improve traffic. A decline may also
reflect seasonality, consolidation into another page, measurement gaps, or
low-volume noise.

The starter `trend_direction` field describes the current window and is only
a provisional proxy. For stronger capstone work, I will investigate a
future-window outcome where features are measured before the target period.
`trend_direction`, `trend_pct`, `content_id`, and `client_id` will not be used
as ordinary model features.

In [ ]:
# Confirm that the public starter data has no obvious raw identifying columns.

private_columns = {
    "client_name",
    "domain",
    "url",
    "page_title",
    "keyword",
    "raw_query"
}

present_private_columns = sorted(private_columns.intersection(df.columns))

print("Private/raw identifying columns found:", present_private_columns)
print(
    "Excluded from ordinary model features:",
    ["content_id", "client_id", "trend_direction", "trend_pct"]
)
print("Only aggregate counts were displayed; no row-level records were published.")

Private/raw identifying columns found: []
Excluded from ordinary model features: ['content_id', 'client_id', 'trend_direction', 'trend_pct']
Only aggregate counts were displayed; no row-level records were published.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.